<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/alex/Experiencia_5_Laboratorio_N%C2%B01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio — Sistema de Monitoreo de Biodiversidad en Reservas Naturales
**Inteligencia Computacional IC415 — RA1 Lab 1 — Experiencia 5** | *Año 2026*

---

> **Archivos de entrada:** `Audio_clasificar.wav` · carpeta `muestras_v2/` · `ambiente_largo.mp3`  
> **Modelo:** Random Forest multi-label con features de energía de banda espectral

### Estructura del pipeline

1. **Sección 0** — Entendimiento del problema y restricciones de hardware  
2. **Sección 1** — Configuración del entorno  
3. **Sección 2** — Análisis técnico del sistema de captura (Nyquist, aliasing)  
4. **Sección 3** — Exploración espectral del audio de entrenamiento  
5. **Sección 4** — Firmas acústicas y extracción de características  
6. **Sección 5** — Construcción del dataset `muestras_v2`  
7. **Sección 6** — Discriminabilidad y reducción de dimensionalidad  
8. **Sección 7** — Modelado y evaluación  
9. **Sección 8** — Eventos simultáneos y clase desconocida  
10. **Sección 9** — Inferencia en campo (`ambiente_largo.mp3`)  
11. **Sección 10** — Viabilidad en hardware embebido  
12. **Sección 11** — Síntesis y recomendaciones  


---
## 0. Entendimiento del problema y restricciones de hardware


In [ ]:
# ── Descripción del sistema y restricciones ──────────────────────────────────
problema = {
    "Tarea":          "Clasificación acústica multi-label (varias especies simultáneas)",
    "Input":          "Ventana de audio de duración fija (grabador autónomo en campo)",
    "Output":         "Log con estampas de tiempo y especies detectadas",
    "Hardware campo": "MCU ~133 MHz, RAM 256 KB, almacenamiento SD",
    "Autonomía req.": "Hasta 6 meses con batería pequeña",
    "Usuario final":  "Biólogos y guardaparques — no ingenieros",
}

especies = {
    "G": ("Grillos alta frecuencia",  "~5 kHz",   "presencia continua nocturna"),
    "C": ("Chicharra",                "~2 kHz",   "canto diurno sostenido"),
    "L": ("Langosta alta frecuencia", "~8–9 kHz", "estridulación intensa"),
}

print('Descripción del sistema:')
for k,v in problema.items():
    print(f'  {k:<20}: {v}')

print('\nEspecies objetivo (prototipo v1.0):')
for code,(nombre,freq,desc) in especies.items():
    print(f'  [{code}] {nombre:<30} {freq:<12} — {desc}')

print('\n> NOTA: El Añapero Castaño (1–4 kHz) y el Yeruvá fueron excluidos del prototipo v1.0.')
print('> Sus rangos espectrales se superponen con Chicharra y Grillos.')
print('> Con el dataset disponible esa ambigüedad degrada las fronteras del clasificador.')


> **¿Por qué estas tres especies?** Se seleccionaron Grillos (~5 kHz), Chicharra (~2 kHz) y Langosta (~8–9 kHz) porque sus firmas espectrales están separadas entre sí por al menos 3 kHz, lo que maximiza la discriminabilidad con un dataset pequeño. El Añapero Castaño (*Lurocalis semitorquatus*) emite en 1–4 kHz, solapándose con ambas clases de insectos. El Yeruvá (*Baryphthengus ruficapillus*) tampoco aparece en el audio de entrenamiento disponible. Ambas especies quedan planificadas para v2.0 del sistema.


---
## 1. Configuración del entorno


In [ ]:
import numpy as np
import scipy.io.wavfile as waves
from scipy.fft import fft, fftfreq
from scipy.signal import spectrogram, butter, sosfilt
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import os, wave, time, pickle, warnings
from collections import Counter
warnings.filterwarnings('ignore')
import zipfile # Import zipfile

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.spines.top': False,
    'axes.spines.right': False,  'figure.dpi': 100,
})

# ── Constantes globales (FIJAS para todo el pipeline) ──────────────────────
ETIQUETAS  = ('G', 'C', 'L')
NOMBRES    = {'G': 'Grillos', 'C': 'Chicharra', 'L': 'Langosta'}
COLORES    = {'G': 'steelblue', 'C': 'crimson', 'L': 'darkorange'}
FREQ_PICO  = {'G': 5000, 'C': 2000, 'L': 8500}

# ── Parámetros FFT ─────────────────────────────────────────────────────────
N_FFT  = 2048
OFFSET = 10
SR_REF = 24000

# ── Rutas ─────────────────────────────────────────────────────────────────
AUDIO_TRAIN   = 'Audio_clasificar.wav'
AUDIO_CAMPO   = 'ambiente_largo.mp3'
PCM_CAMPO     = 'ambiente_largo_raw.pcm'  # se genera en Sección 2
ZIP_FILE_PATH = 'muestras.zip'
MUESTRAS_PATH = 'muestras_v2' # This will be the directory where the zip is extracted

# Check if MUESTRAS_PATH directory exists, if not, unzip the ZIP_FILE_PATH
if not os.path.exists(MUESTRAS_PATH):
    if os.path.exists(ZIP_FILE_PATH):
        with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
            zip_ref.extractall(MUESTRAS_PATH) # Extract into the MUESTRAS_PATH directory
        print(f"'{ZIP_FILE_PATH}' descomprimido en '{MUESTRAS_PATH}'.")
    else:
        print(f"Advertencia: El archivo zip '{ZIP_FILE_PATH}' no se encontró.")
        print(f"No se pudo crear el directorio '{MUESTRAS_PATH}'.")
elif not os.path.isdir(MUESTRAS_PATH): # If it exists but is not a directory
    print(f"Advertencia: '{MUESTRAS_PATH}' existe pero no es un directorio.")


print(f'Entorno configurado.')
print(f'Clases: {ETIQUETAS}')
print(f'N_FFT={N_FFT}  OFFSET={OFFSET}  SR_REF={SR_REF} Hz')

archivos_v2 = []
if os.path.exists(MUESTRAS_PATH) and os.path.isdir(MUESTRAS_PATH): # Now check for the directory
    archivos_v2 = sorted([f for f in os.listdir(MUESTRAS_PATH) if f.endswith('.wav')])
    print(f'Archivos en {MUESTRAS_PATH}/: {len(archivos_v2)}')
else:
    print(f"No se pudieron listar archivos en '{MUESTRAS_PATH}' porque el directorio no existe.")

---
## 2. Análisis técnico del sistema de captura

**Premisa:** *¿Garantiza la configuración digital el registro íntegro de todas las especies? ¿Cuál es el límite teórico de frecuencia? ¿Qué fenómeno ocurre si se lo supera?*


In [ ]:
!wget -c --no-check-certificate "https://drive.google.com/uc?export=download&id=1bRvDjRE5AJzJ4MqMGlPbcA0nn8Vgi7Vl&confirm=t" -O Audio_clasificar.zip

In [ ]:
!unzip Audio_clasificar.zip

Archive:  Audio_clasificar.zip
replace Audio_clasificar.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
# ── Propiedades del archivo de entrenamiento ──────────────────────────────
fs, audio_completo = waves.read(AUDIO_TRAIN)
duracion_s = len(audio_completo) / fs

print('=' * 55)
print('PROPIEDADES TÉCNICAS — Audio_clasificar.wav')
print('=' * 55)
print(f'  Frecuencia de muestreo (fs):  {fs:,} Hz')
print(f'  Número de muestras:           {len(audio_completo):,}')
print(f'  Duración total:               {duracion_s:.2f} s')
print(f'  Canales:                      {1 if audio_completo.ndim==1 else audio_completo.shape[1]} (mono)')
print(f'  Tipo de dato:                 {audio_completo.dtype}')
print()
print(f'  TEOREMA DE NYQUIST:')
print(f'  Frecuencia de Nyquist (fs/2): {fs//2:,} Hz')
print()
print('  COBERTURA DE ESPECIES:')
for code, freq in FREQ_PICO.items():
    ok = '✓ CUBIERTA' if freq < fs//2 else '✗ FUERA DE RANGO'
    print(f'  [{code}] {NOMBRES[code]:<20} ~{freq//1000} kHz  → {ok}')


> **Teorema de Nyquist:** Para representar fielmente una señal de frecuencia $f$, la frecuencia de muestreo debe ser $f_s \geq 2f$. Con $f_s = 24\,000$ Hz, la frecuencia de Nyquist es **12 000 Hz**. Todas las especies objetivo (Grillos ≈5 kHz, Chicharra ≈2 kHz, Langosta ≈8–9 kHz) están muy por debajo de ese límite.

> **Aliasing:** Si existiera una especie con canto por encima de 12 kHz, sus componentes se *reflejarían* hacia frecuencias más bajas (por ejemplo, 14 kHz aparecería como 24k − 14k = 10 kHz), contaminando las firmas de otras especies. Con las tres especies actuales el sistema es completamente adecuado.


In [ ]:
# ── Resolución espectral y conversión del audio de campo ───────────────────
freq_res = fs / N_FFT
print(f'RESOLUCIÓN ESPECTRAL:')
print(f'  N_FFT = {N_FFT} puntos → {freq_res:.2f} Hz/bin')
print(f'  Ventana de análisis: {N_FFT/fs*1000:.1f} ms')
print()
for code, freq in FREQ_PICO.items():
    print(f'  [{code}] {NOMBRES[code]:<15} ~{freq} Hz → bin #{int(freq/freq_res)}')

# Convertir ambiente_largo.mp3 a PCM s16le 24 kHz mono
import subprocess
ret = subprocess.run(
    ['ffmpeg', '-i', AUDIO_CAMPO, '-ar', str(SR_REF), '-ac', '1',
     '-f', 's16le', PCM_CAMPO, '-y'],
    capture_output=True
)
audio_campo = np.frombuffer(open(PCM_CAMPO,'rb').read(), dtype=np.int16)
dur_campo = len(audio_campo) / SR_REF
print(f'\nAudio de campo decodificado:')
print(f'  Duración: {dur_campo:.0f} s ({dur_campo/60:.1f} min)')
print(f'  Muestras: {len(audio_campo):,}  |  fs={SR_REF} Hz')


---
## 3. Exploración espectral del audio de entrenamiento

**Premisa:** *¿Por qué trabajar en el dominio de la frecuencia en lugar del tiempo?*


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# ── Onda temporal ─────────────────────────────────────────────────────────
t_eje = np.linspace(0, duracion_s, len(audio_completo))
axes[0].plot(t_eje, audio_completo, color='steelblue', lw=0.4, alpha=0.8)
axes[0].set_title('Forma de onda — Audio_clasificar.wav', fontsize=12)
axes[0].set_ylabel('Amplitud (int16)')
axes[0].set_xlabel('Tiempo (s)')

# ── Espectrograma ─────────────────────────────────────────────────────────
f_sg, t_sg, Sxx = spectrogram(audio_completo.astype(float), fs, nperseg=1024, noverlap=512)
im = axes[1].pcolormesh(t_sg, f_sg/1000, 10*np.log10(Sxx+1e-10),
                         shading='gouraud', cmap='inferno', vmin=-60, vmax=40)
axes[1].set_ylim(0, 12)
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('Frecuencia (kHz)')
axes[1].set_title('Espectrograma — Audio_clasificar.wav', fontsize=12)
for code, freq in FREQ_PICO.items():
    axes[1].axhline(freq/1000, color=COLORES[code], linestyle='--', lw=1.2,
                    label=f'[{code}] {NOMBRES[code]} ~{freq//1000} kHz')
axes[1].legend(loc='upper right', fontsize=9)
plt.colorbar(im, ax=axes[1], label='dB')
plt.tight_layout()
plt.savefig('espectrograma_completo.png', dpi=100, bbox_inches='tight')
plt.show()


> **¿Por qué dominio frecuencia?** La señal temporal dice *cuándo* hay energía, no *qué frecuencia* tiene. Dos insectos cantando a la vez son indistinguibles en el tiempo si tienen amplitudes similares. La FFT revela *dónde* en el espectro se concentra la energía de cada especie: Grillos en ~5 kHz, Chicharra en ~2 kHz, Langosta en ~8–9 kHz. Estas posiciones son estables independientemente de la distancia al grabador.

> **¿Por qué la amplitud absoluta no sirve?** Un grillo a 5 m produce la misma *forma* espectral que uno a 50 m, solo con menor amplitud. Usar amplitud como feature haría que el mismo animal clasificara diferente según su distancia. La distribución de energía *relativa* entre bandas es invariante a distancia, viento y ganancia del micrófono.


---
## 4. Firmas acústicas y extracción de características

**Premisa:** *¿Qué características resultaron más discriminatorias? ¿Por qué la amplitud no es confiable?*


In [ ]:
# ── Función de extracción de características ──────────────────────────────
# Se definen 7 BANDAS ESPECTRALES que cubren 0–12 kHz
# Para cada ventana se calcula la energía en cada banda, normalizada por la energía total
# → 7 valores que suman 1 (distribución relativa de energía)
# + 3 valores adicionales: posición del pico dentro de la banda de cada especie
# Total: 10 features / ventana

BANDAS_SPEC = [
    ('ruido_bajo',   0,    1500),   # ruido ambiental / respiración
    ('C',         1700,   2400),   # Chicharra  ~2 kHz
    ('medio_bajo', 2500,  4000),   # zona intermedia
    ('G',         4200,   5800),   # Grillo     ~5 kHz
    ('medio_alto', 5900,  7400),   # zona intermedia alta
    ('L',         7500,   9500),   # Langosta   ~8–9 kHz
    ('alto',      9500,  12000),   # ruido de alta frecuencia
]

def extraer_features(signal, n_fft=N_FFT, sr=SR_REF):
    """
    Extrae 10 features de una señal de audio:
      · 7 energías de banda normalizadas (suma = 1) → invariantes a amplitud
      · 3 posiciones del pico espectral (G, C, L) en su banda respectiva

    La normalización por energía total hace al vector INVARIANTE
    a la distancia del animal al grabador.
    """
    s = signal.astype(float)
    if len(s) < n_fft: s = np.pad(s, (0, n_fft - len(s)))
    else: s = s[:n_fft]
    esp   = np.abs(fft(s, n=n_fft))[:n_fft//2]
    freqs = fftfreq(n_fft, 1/sr)[:n_fft//2]

    # Energías de banda
    energias = [np.sum(esp[(freqs>=lo) & (freqs<=hi)]**2) for _,lo,hi in BANDAS_SPEC]
    total = sum(energias) + 1e-10
    feat_e = [e/total for e in energias]   # normalizadas

    # Posición del pico en banda de cada especie (normalizado a [0,1])
    feat_p = []
    for lo, hi in [(4200,5800), (1700,2400), (7500,9500)]:
        mask = (freqs >= lo) & (freqs <= hi)
        if mask.any():
            f_pico = freqs[mask][np.argmax(esp[mask])]
            feat_p.append((f_pico - lo) / (hi - lo + 1e-10))
        else:
            feat_p.append(0.5)

    return np.array(feat_e + feat_p, dtype=np.float32)

# Vector de frecuencias para gráficos
freqs_eje = fftfreq(N_FFT, 1/SR_REF)[OFFSET:N_FFT//2]

print('Función extraer_features definida.')
print(f'Output: {len(BANDAS_SPEC)+3} features por ventana')
print(f'  [{len(BANDAS_SPEC)} energías de banda + 3 posiciones de pico]')


In [ ]:
# ── Firmas espectrales promedio por clase pura (de muestras_v2) ───────────
def leer_wav(path):
    with wave.open(path, 'rb') as wf:
        raw = wf.readframes(wf.getnframes())
    return np.frombuffer(raw, dtype=np.int16).astype(np.float32)

def fft_espectro(signal, n=N_FFT, offset=OFFSET):
    s = signal.astype(float)
    if len(s) < n: s = np.pad(s, (0, n-len(s)))
    else: s = s[:n]
    return np.abs(fft(s, n=n))[offset:n//2]

# Solo muestras originales (sin AUG) de clases puras
CLASES_PURAS = {c: [] for c in 'GCL'}
for nombre in archivos_v2:
    if '_AUG_' in nombre: continue
    prefijo = nombre.split('_')[0]
    if prefijo in CLASES_PURAS:
        sig = leer_wav(os.path.join(MUESTRAS_PATH, nombre))
        CLASES_PURAS[prefijo].append(fft_espectro(sig))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, code in zip(axes, 'GCL'):
    arr  = np.array(CLASES_PURAS[code])
    mu, sd = arr.mean(axis=0), arr.std(axis=0)
    ax.fill_between(freqs_eje/1000, mu-sd, mu+sd, alpha=0.25, color=COLORES[code])
    ax.plot(freqs_eje/1000, mu, color=COLORES[code], lw=1.5,
            label=f'Media (n={len(CLASES_PURAS[code])})')
    ax.axvline(FREQ_PICO[code]/1000, color=COLORES[code], linestyle='--',
               lw=1, alpha=0.7, label=f'Pico ~{FREQ_PICO[code]//1000} kHz')
    ax.set_title(f'[{code}] {NOMBRES[code]}', fontsize=11)
    ax.set_xlabel('Frecuencia (kHz)')
    ax.set_ylabel('Magnitud FFT')
    ax.set_xlim(0, 12)
    ax.legend(fontsize=8)
plt.suptitle('Firmas espectrales promedio por clase (banda ±1σ)', fontsize=12)
plt.tight_layout()
plt.savefig('firmas_espectrales.png', dpi=100, bbox_inches='tight')
plt.show()
print('Guardado: firmas_espectrales.png')


> **Lo que el modelo 'escucha':** El clasificador no usa la amplitud absoluta sino la *distribución relativa de energía por bandas de frecuencia*. Cada ventana de 500 ms se convierte en un vector de 10 valores: 7 proporciones de energía (una por banda) y 3 posiciones de pico (una por especie). Este vector es el que el Random Forest usa para clasificar.

> **Ventaja respecto al FFT completo (1014 bins):** Las 10 features de banda concentran toda la información discriminatoria en un vector 100× más pequeño, lo que acelera el entrenamiento, reduce el riesgo de sobreajuste con pocos datos, y resulta directamente interpretable para ingenieros y biólogos.


---
## 5. Construcción del dataset `muestras_v2`

**Premisa:** *¿Cómo se transformó un audio de 25 s en un conjunto de muestras de entrenamiento? ¿Qué criterios definen el inicio y fin de un evento?*


### 5.1 Por qué se descartó el dataset original

El dataset original (52 muestras extraídas manualmente en Audacity) presentaba un problema crítico de **distribución shift**: las muestras de Chicharra y Langosta eran segmentos *limpios* extraídos con reducción de ruido, donde esa especie aparecía *sola*. En producción, sin embargo, los Grillos están presentes **en el 100% del audio de campo** (son la especie de fondo constante). El modelo nunca había aprendido a detectar 'Chicharra con Grillos de fondo', y por eso clasificaba todo como G.

El resultado previo lo confirmaba: G=87,7%, C=1,7%, L=0,3%.

### 5.2 Metodología de construcción de `muestras_v2`

El nuevo dataset fue construido directamente desde el audio de entrenamiento, usando análisis espectral automatizado para identificar qué especie está activa en cada segmento de 500 ms. El proceso consistió en:

1. **Segmentación con ventana deslizante:** ventanas de 500 ms con hop de 100 ms sobre el audio completo.
2. **Etiquetado automático por energía de banda:** para cada ventana, se calculó la energía normalizada en las bandas G, C y L. Se asignó presencia de la especie si la energía superaba un umbral (G>0,30 / C>0,40 / L>0,22).
3. **Clases resultantes:** G puro (t=12–25 s, sin C ni L), GC (mezcla natural), GCL (las tres juntas), GL (G+L sin C).
4. **C y L puros:** generados por filtrado paso-banda (Butterworth orden 6) de segmentos ricos en esa especie. Esto enseña al modelo la firma espectral pura de cada una sin ruido de fondo.
5. **Data augmentation:** 10 variantes por muestra con ruido blanco gaussiano (SNR 7–20 dB), ruido rosa (1/f) y variaciones de amplitud. El ruido rosa es el más realista para ambientes naturales.

| Clase | Origen | Orig. | Total c/aug |
|-------|--------|------:|-----------:|
| G | Segmentos reales t=12–25s | 126 | 630 |
| GC | Segmentos reales (C+G) t=2–12s | 59 | 472 |
| GCL | Segmentos reales (tres juntas) | 17 | 306 |
| GL | Segmentos reales (G+L) | 2 | 36 |
| C | Bandpass 1700–2400 Hz de GC | 30 | 540 |
| L | Bandpass 7500–9500 Hz de GCL | 12 | 216 |
| CL | C filtrada + L filtrada | 12 | 216 |
| **TOTAL** | | **258** | **2416** |


In [ ]:
# ── Parser de etiquetas multi-label ──────────────────────────────────────
def parsear_etiqueta(nombre_archivo, etiquetas=ETIQUETAS):
    """
    Extrae el vector multi-hot a partir del nombre del archivo.
    Convención: el prefijo antes del primer '_' contiene las clases presentes.
    Ejemplos: 'GC_001.wav' → [1,1,0]  |  'L_005_AUG_WN12.wav' → [0,0,1]
    """
    prefijo = nombre_archivo.split('_')[0]
    return [1 if e in prefijo else 0 for e in etiquetas]

# Verificación
tests = {'G_001.wav':[1,0,0],'C_003.wav':[0,1,0],'L_005.wav':[0,0,1],
         'GC_001.wav':[1,1,0],'GCL_001.wav':[1,1,1],'GL_002.wav':[1,0,1]}
print(f'Verificación del parser (ETIQUETAS={ETIQUETAS}):')
todos_ok = True
for fname, esp in tests.items():
    ob = parsear_etiqueta(fname)
    ok = ob == esp
    todos_ok &= ok
    print(f'  {fname:<20} → {ob}  {"✓" if ok else "✗"}')
print(f'Parser: {"CORRECTO ✓" if todos_ok else "ERRORES ✗"}')


In [ ]:
# ── Carga del dataset completo ────────────────────────────────────────────
X_list, y_list = [], []
for nombre in archivos_v2:
    sig = leer_wav(os.path.join(MUESTRAS_PATH, nombre))
    X_list.append(extraer_features(sig))
    y_list.append(parsear_etiqueta(nombre))

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int8)

print(f'Dataset cargado:')
print(f'  Muestras totales:     {X.shape[0]}')
print(f'  Features por muestra: {X.shape[1]} (energías de banda + picos)')
print(f'  Etiquetas shape:      {y.shape}')
print()
print('Distribución por clase (incluyendo mezclas):')
for i, code in enumerate(ETIQUETAS):
    n_pos = int(y[:,i].sum())
    pct   = n_pos / len(y) * 100
    print(f'  [{code}] {NOMBRES[code]:<12}: {n_pos:>4} positivos ({pct:.1f}%)')


> **¿Cómo se decidió dónde comienza y termina un evento?** Se usó un criterio cuantitativo: una ventana tiene presencia de la especie X si la energía normalizada en la banda X supera un umbral empírico (0,30 para G, 0,40 para C, 0,22 para L). El umbral de L es más bajo porque Langosta tiene menor intensidad relativa en el audio. La ventana deslizante de 500 ms con hop de 100 ms garantiza que incluso eventos breves (>200 ms) queden representados en al menos una ventana.


---
## 6. Discriminabilidad y reducción de dimensionalidad

**Premisa:** *¿Las firmas de banda permiten separar las clases? ¿Es posible reducir la dimensionalidad sin perder información?*


In [ ]:
# ── Visualización de features por clase ──────────────────────────────────
nombres_feat = [b[0] for b in BANDAS_SPEC] + ['pico_G_norm', 'pico_C_norm', 'pico_L_norm']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, code, idx in zip(axes, ETIQUETAS, range(3)):
    mask_pos = y[:,idx] == 1
    mask_neg = y[:,idx] == 0
    mu_pos = X[mask_pos].mean(axis=0)
    mu_neg = X[mask_neg].mean(axis=0)
    x_pos = np.arange(len(nombres_feat))
    ax.bar(x_pos - 0.2, mu_pos[:len(nombres_feat)], 0.4,
           label=f'{NOMBRES[code]} presente', color=COLORES[code], alpha=0.8)
    ax.bar(x_pos + 0.2, mu_neg[:len(nombres_feat)], 0.4,
           label='Ausente', color='gray', alpha=0.5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(nombres_feat[:len(nombres_feat)], rotation=45, ha='right', fontsize=7)
    ax.set_title(f'[{code}] {NOMBRES[code]} — features medias', fontsize=10)
    ax.legend(fontsize=7)
    ax.set_ylabel('Valor medio')
plt.suptitle('Diferencias en features entre muestras con/sin cada especie', fontsize=11)
plt.tight_layout()
plt.savefig('discriminabilidad_features.png', dpi=100, bbox_inches='tight')
plt.show()
print('Guardado: discriminabilidad_features.png')


In [ ]:
# ── PCA: cuántos componentes capturan la varianza relevante ───────────────
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler_pca = StandardScaler()
X_scaled   = scaler_pca.fit_transform(X)

pca_full = PCA()
pca_full.fit(X_scaled)
var_acum = np.cumsum(pca_full.explained_variance_ratio_)

n90 = np.argmax(var_acum >= 0.90) + 1
n95 = np.argmax(var_acum >= 0.95) + 1
n99 = np.argmax(var_acum >= 0.99) + 1
print(f'Varianza explicada acumulada:')
print(f'  90% con {n90} componentes (de {X.shape[1]})')
print(f'  95% con {n95} componentes')
print(f'  99% con {n99} componentes')
print(f'  → Con solo {X.shape[1]} features, PCA no aporta reducción significativa.')
print(f'  → Las 10 features de banda ya son una representación compacta y relevante.')

# Proyección 2D para visualizar separabilidad
pca2 = PCA(n_components=2)
X_2d = pca2.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, code, idx in zip(axes, ETIQUETAS, range(3)):
    mask1 = y[:,idx] == 1
    mask0 = y[:,idx] == 0
    ax.scatter(X_2d[mask0,0], X_2d[mask0,1], c='lightgray', s=4, alpha=0.4, label='Ausente')
    ax.scatter(X_2d[mask1,0], X_2d[mask1,1], c=COLORES[code], s=6, alpha=0.6,
               label=f'{NOMBRES[code]}')
    ax.set_title(f'PCA 2D — [{code}] {NOMBRES[code]}', fontsize=10)
    ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
    ax.legend(fontsize=8)
plt.suptitle('Separabilidad en el espacio PCA 2D', fontsize=11)
plt.tight_layout()
plt.savefig('pca_separabilidad.png', dpi=100, bbox_inches='tight')
plt.show()


> **¿Por qué con 10 features el PCA no aporta gran reducción?** Porque el espacio de 10 dimensiones ya es compacto. El análisis muestra que se necesitan pocos componentes principales para capturar el 95% de la varianza, lo que confirma que las 10 features de banda son relevantes y sin redundancia excesiva.

> **Ventaja operativa en hardware embebido:** 10 features vs. 1014 (FFT completo) representa una reducción de 100× en el vector de entrada al clasificador. Esto reduce el tiempo de inferencia y la memoria necesaria, haciéndolo viable en microcontroladores de bajo consumo.


---
## 7. Modelado y evaluación

**Premisa:** *¿Qué modelo y métricas se eligen? ¿Cómo se evalúa el desempeño?*


In [ ]:
# ── División train/test estratificada ────────────────────────────────────
# Estratificación por G (clase más balanceada)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y[:,0]
)
print(f'Train: {len(X_tr)} muestras | Test: {len(X_te)} muestras')

# ── Entrenar un RF por especie ────────────────────────────────────────────
# Clasificadores independientes para G, C y L:
# · Permite ajustar umbrales de probabilidad por especie
# · class_weight='balanced_subsample': compensa el desbalance
# · n_estimators=300: suficiente para estabilidad con 2416 muestras

MODELOS = {}
for i, code in enumerate(ETIQUETAS):
    rf = RandomForestClassifier(
        n_estimators=300, max_depth=None,
        class_weight='balanced_subsample',
        random_state=SEED, n_jobs=-1
    )
    rf.fit(X_tr, y_tr[:, i])
    MODELOS[code] = rf

print('Modelos entrenados: un Random Forest por especie.')
print('  Justificación: RF maneja bien el desbalance (balanced_subsample),')
print('  no requiere normalización previa, y produce probabilidades')
print('  calibradas que permiten ajustar umbrales de detección.')


In [ ]:
# ── Evaluación en conjunto de test ───────────────────────────────────────
UMBRALES = {'G': 0.50, 'C': 0.15, 'L': 0.12}
# Los umbrales de C y L son más bajos que el defecto (0.5) porque:
# · C y L aparecen siempre con G de fondo → sus probabilidades son más bajas
# · El costo de un falso negativo (no detectar una especie presente) supera
#   el costo de un falso positivo para un sistema de monitoreo

y_pred_prob = {}
y_pred_bin  = {}
for code in ETIQUETAS:
    probas = MODELOS[code].predict_proba(X_te)
    y_pred_prob[code] = probas[:,1] if probas.shape[1]>1 else np.ones(len(X_te))
    y_pred_bin[code]  = (y_pred_prob[code] >= UMBRALES[code]).astype(int)

y_pred = np.column_stack([y_pred_bin[c] for c in ETIQUETAS])
y_true = y_te

print('=' * 60)
print('DESEMPEÑO POR ESPECIE EN CONJUNTO DE TEST')
print('=' * 60)
print(f'{"Especie":<12}  {"Umbral":>7}  {"P":>6}  {"R":>6}  {"F1":>6}  {"Soporte":>8}')
print('-' * 55)
for i, code in enumerate(ETIQUETAS):
    pr = precision_score(y_true[:,i], y_pred[:,i], zero_division=0)
    re = recall_score(y_true[:,i],    y_pred[:,i], zero_division=0)
    f1 = f1_score(y_true[:,i],         y_pred[:,i], zero_division=0)
    soporte = int(y_true[:,i].sum())
    print(f'  [{code}] {NOMBRES[code]:<10} th={UMBRALES[code]:.2f}  {pr:6.3f}  {re:6.3f}  {f1:6.3f}  {soporte:8d}')

f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f'\n  F1-macro: {f1_macro:.3f}')

# Precisión exacta (todas las etiquetas correctas)
exact = np.all(y_pred == y_true, axis=1).mean()
print(f'  Exact match: {exact:.3f}')


In [ ]:
# ── Matrices de confusión ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, code, idx in zip(axes, ETIQUETAS, range(3)):
    cm = confusion_matrix(y_true[:,idx], y_pred[:,idx])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Ausente','Presente'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'[{code}] {NOMBRES[code]}  (th={UMBRALES[code]:.2f})', fontsize=10)
plt.suptitle('Matrices de confusión — conjunto de test', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# ── Importancia de features ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, code in zip(axes, ETIQUETAS):
    importances = MODELOS[code].feature_importances_
    ax.bar(range(len(nombres_feat)), importances, color=COLORES[code], alpha=0.8)
    ax.set_xticks(range(len(nombres_feat)))
    ax.set_xticklabels(nombres_feat, rotation=45, ha='right', fontsize=7)
    ax.set_title(f'[{code}] Importancia de features', fontsize=10)
    ax.set_ylabel('Importancia (Gini)')
plt.suptitle('Importancia de features por clasificador', fontsize=11)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print('La banda específica de cada especie es, como se esperaba, la más importante.')


In [ ]:
# ── Guardar modelo ────────────────────────────────────────────────────────
import pickle
modelo_exportado = {'modelos': MODELOS, 'umbrales': UMBRALES,
                    'etiquetas': ETIQUETAS, 'bandas': BANDAS_SPEC}
with open('modelo_biodiversidad.pkl', 'wb') as f:
    pickle.dump(modelo_exportado, f)

size_kb = os.path.getsize('modelo_biodiversidad.pkl') / 1024
print(f'Modelo guardado: modelo_biodiversidad.pkl ({size_kb:.1f} KB)')


> **¿Por qué Random Forest?** Se eligió por tres razones: (1) maneja naturalmente el desbalance de clases con `balanced_subsample`; (2) produce probabilidades calibradas que permiten ajustar el umbral de detección según el trade-off falso positivo / falso negativo deseado; (3) el tamaño del modelo en disco es pequeño (~KB) y la inferencia es determinista y rápida en hardware embebido.

> **¿Por qué umbrales distintos para cada especie?** El clasificador de C devuelve probabilidades más bajas que el de G porque C aparece siempre con G de fondo. Usar th=0,50 para C perdería la mitad de las detecciones reales. El umbral bajo (0,15–0,12) aumenta el recall a costa de más falsos positivos, lo cual es preferible en un sistema de monitoreo donde *no detectar* una especie es más costoso que reportar una detección incierta.


---
## 8. Eventos simultáneos y clase desconocida

**Premisa:** *¿Puede el sistema manejar varias especies simultáneas? ¿Qué ocurre ante sonidos no entrenados?*


In [ ]:
# ── Capacidad multi-label ─────────────────────────────────────────────────
# El sistema usa clasificadores INDEPENDIENTES por especie:
# G, C y L se detectan en paralelo, sin excluirse mutuamente

# Verificar que muestras GCL se detectan con los tres = 1
archivos_gcl = [f for f in archivos_v2 if f.startswith('GCL') and '_AUG_' not in f]
predicciones_gcl = []
for nombre in archivos_gcl[:5]:
    sig  = leer_wav(os.path.join(MUESTRAS_PATH, nombre))
    feat = extraer_features(sig)
    pred = {c: int(MODELOS[c].predict_proba(feat.reshape(1,-1))[0,1] >= UMBRALES[c])
            for c in ETIQUETAS}
    predicciones_gcl.append((nombre, pred))

print('Detección en muestras GCL (todas deberían detectar G=1, C=1, L=1):')
for nombre, pred in predicciones_gcl:
    print(f'  {nombre:<25} → G={pred["G"]}  C={pred["C"]}  L={pred["L"]}')

print()
print('Ventajas del enfoque multi-label:')
print('  · Detecta varias especies simultáneas en la misma ventana de 500 ms')
print('  · Cada clasificador es independiente: detectar G no impide detectar C')
print('  · El log puede registrar co-ocurrencias, dato valioso para ecología')


In [ ]:
# ── Comportamiento ante clase desconocida ─────────────────────────────────
np.random.seed(99)
ruido_puro = np.random.randn(N_FFT).astype(np.float32) * 1000
feat_ruido = extraer_features(ruido_puro)
pred_ruido = {c: MODELOS[c].predict_proba(feat_ruido.reshape(1,-1))[0,1] for c in ETIQUETAS}
pred_bin   = {c: int(pred_ruido[c] >= UMBRALES[c]) for c in ETIQUETAS}

print('CLASE DESCONOCIDA — input: ruido blanco gaussiano puro')
print('(No corresponde a ninguna especie entrenada)')
for c in ETIQUETAS:
    print(f'  [{c}] probabilidad={pred_ruido[c]:.3f}  detectado={bool(pred_bin[c])}')

print()
print('El sistema actual es CLOSED-WORLD: si el input no corresponde')
print('a ninguna clase, el modelo fuerza la predicción más cercana.')
print()
print('ESTRATEGIAS PROPUESTAS PARA v2.0:')
print('  1. Filtro de energía: RMS < umbral → no clasificar (silencio o ruido)')
print('  2. Umbral de confianza: si max(proba) < 0.10 → reportar "Desconocido"')
print('  3. Clase Fondo/Desconocido en entrenamiento con muestras de lluvia/viento')
print('  4. Confirmación temporal: N ventanas consecutivas antes de loguear')


> **Eventos simultáneos:** El enfoque de clasificadores independientes por especie permite detectar múltiples especies en la misma ventana de 500 ms. El vector de predicción puede contener [1,1,1] cuando las tres están presentes. Esto es esencial en la selva, donde el silencio absoluto es la excepción.

> **Clase desconocida:** El modelo actual es *closed-world* — no puede decir 'no sé'. La estrategia de mitigación principal para v1.0 es el filtro de energía: ventanas con RMS muy bajo (por debajo de umbral empírico) se descartan antes de clasificar. Para v2.0, agregar muestras de lluvia, viento y actividad humana como clase 'Fondo/Desconocido' resuelve el problema correctamente.


---
## 9. Inferencia y validación en campo (`ambiente_largo.mp3`)

**Premisa:** *¿El desempeño en condiciones reales es coherente con el entrenamiento? ¿Qué diferencias se observan y a qué se atribuyen?*


In [ ]:
# ── Función de inferencia sobre audio largo ───────────────────────────────
def procesar_audio_campo(audio_signal, fs, modelos, etiquetas, umbrales,
                          duracion_ventana_s=0.5, solapamiento_s=0.25,
                          umbral_rms=200):
    """
    Procesa un audio largo con ventana deslizante.
    Vectoriza la extracción de features (más eficiente que window-by-window).
    Aplica umbrales de probabilidad por especie.
    """
    WIN = int(duracion_ventana_s * fs)
    HOP = int(solapamiento_s * fs)
    starts = list(range(0, len(audio_signal) - WIN, HOP))

    # Precomputar features en batch
    feat_all = np.array([extraer_features(audio_signal[s:s+WIN]) for s in starts])

    t0 = time.perf_counter()
    # Probas por especie (batch)
    probas = {}
    for c in etiquetas:
        p = modelos[c].predict_proba(feat_all)
        probas[c] = p[:,1] if p.shape[1] > 1 else np.ones(len(feat_all))
    t_inf = (time.perf_counter() - t0) * 1000

    log = []
    n_sil = 0
    for i, start in enumerate(starts):
        vent = audio_signal[start:start+WIN]
        rms  = float(np.sqrt(np.mean(vent.astype(float)**2)))
        if rms < umbral_rms:
            n_sil += 1
            continue
        spp = [c for c in etiquetas if probas[c][i] >= umbrales[c]]
        if spp:
            log.append({'t_ini': start/fs, 't_fin': (start+WIN)/fs,
                        'especies': spp, 'rms': rms,
                        'pG': float(probas['G'][i]),
                        'pC': float(probas['C'][i]),
                        'pL': float(probas['L'][i])})

    stats = {
        'ventanas_total':      len(starts),
        'ventanas_silencio':   n_sil,
        'ventanas_procesadas': len(starts) - n_sil,
        'detecciones':         len(log),
        'infer_total_ms':      t_inf,
        'infer_ms_por_vent':   t_inf / max(1, len(starts)),
        'probas':              probas,
    }
    return log, stats

print('Función procesar_audio_campo definida.')
print('Procesando', len(audio_campo)//SR_REF, 's de audio de campo...')


In [ ]:
# ── Procesar audio completo ───────────────────────────────────────────────
log, stats = procesar_audio_campo(
    audio_signal       = audio_campo,
    fs                 = SR_REF,
    modelos            = MODELOS,
    etiquetas          = ETIQUETAS,
    umbrales           = UMBRALES,
    duracion_ventana_s = 0.5,
    solapamiento_s     = 0.25,
    umbral_rms         = 200,
)

print('=' * 55)
print('ESTADÍSTICAS DE PROCESAMIENTO')
print('=' * 55)
print(f'  Ventanas totales:    {stats["ventanas_total"]:>5}')
print(f'  Ventanas en silencio:{stats["ventanas_silencio"]:>5} (filtradas)')
print(f'  Ventanas clasificadas:{stats["ventanas_procesadas"]:>4}')
print(f'  Detecciones en log:  {stats["detecciones"]:>5}')
print(f'  Inferencia total:    {stats["infer_total_ms"]:>7.0f} ms')
print(f'  Inferencia/ventana:  {stats["infer_ms_por_vent"]:>7.3f} ms')
print()

cnt = Counter(sp for e in log for sp in e['especies'])
proc = stats['ventanas_procesadas']
print('Detecciones por especie:')
for code in ETIQUETAS:
    n   = cnt[code]
    pct = n / proc * 100 if proc > 0 else 0
    print(f'  [{code}] {NOMBRES[code]:<12}: {n:>5} ventanas ({pct:.1f}% del audio)')


In [ ]:
# ── Log textual (primeras 30 entradas con C o L) ───────────────────────────
print('LOG DE DETECCIONES — primeras 30 entradas con C o L presentes')
print(f'{"t_ini":>8}  {"t_fin":>6}  Especies detectadas             probas')
print('-' * 70)
n_mostradas = 0
for entry in log:
    if 'C' in entry['especies'] or 'L' in entry['especies']:
        nombres_det = ' + '.join(NOMBRES[sp] for sp in entry['especies'])
        print(f"{entry['t_ini']:>8.2f}s  "
              f"{entry['t_fin']:>6.2f}s  "
              f"{nombres_det:<35} "
              f"pC={entry['pC']:.2f} pL={entry['pL']:.2f}")
        n_mostradas += 1
        if n_mostradas >= 30: break
if len(log) > n_mostradas:
    print(f'  ... ({len(log) - n_mostradas} entradas adicionales en el log completo)')


In [ ]:
# ── Guardar log completo en CSV ───────────────────────────────────────────
import csv
with open('log_detecciones_campo.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['t_ini','t_fin','especies','rms','pG','pC','pL'])
    writer.writeheader()
    for e in log:
        writer.writerow({**e, 'especies': '+'.join(e['especies'])})
print(f'Log guardado: log_detecciones_campo.csv ({len(log)} entradas)')


In [ ]:
# ── Espectrograma con detecciones superpuestas (primeros 420 s) ───────────
VENTANA_VIS = 420
audio_vis   = audio_campo[:int(VENTANA_VIS * SR_REF)]

fig, ax = plt.subplots(figsize=(18, 5))
f_sg, t_sg, Sxx = spectrogram(audio_vis.astype(float), SR_REF,
                               nperseg=1024, noverlap=512)
ax.pcolormesh(t_sg, f_sg/1000, 10*np.log10(Sxx+1e-10),
              shading='gouraud', cmap='inferno', vmin=-60, vmax=40, alpha=0.85)

for entry in log:
    if entry['t_fin'] > VENTANA_VIS: break
    for sp in entry['especies']:
        ax.plot([entry['t_ini'], entry['t_fin']],
                [FREQ_PICO[sp]/1000, FREQ_PICO[sp]/1000],
                color=COLORES[sp], linewidth=3, alpha=0.7, solid_capstyle='round')

leyenda = [Line2D([0],[0], color=COLORES[c], linewidth=3,
                  label=f'[{c}] {NOMBRES[c]} (~{FREQ_PICO[c]//1000} kHz)')
           for c in ETIQUETAS]
ax.legend(handles=leyenda, fontsize=9, loc='upper right')
ax.set_xlabel('Tiempo (s)', fontsize=11)
ax.set_ylabel('Frecuencia (kHz)', fontsize=11)
ax.set_ylim(0, 12)
ax.set_title(
    f'Log de detecciones — ambiente_largo.mp3 (primeros {VENTANA_VIS} s)\n'
    '(líneas = detecciones del sistema · fondo = espectrograma)',
    fontsize=11
)
plt.tight_layout()
plt.savefig('log_detecciones_campo.png', dpi=100, bbox_inches='tight')
plt.show()
print('Guardado: log_detecciones_campo.png')


In [ ]:
# ── Análisis temporal: actividad acumulada por minuto ─────────────────────
dur_min = int(len(audio_campo) / SR_REF / 60)
act_por_min = {c: np.zeros(dur_min) for c in ETIQUETAS}
for entry in log:
    min_idx = int(entry['t_ini'] / 60)
    if min_idx < dur_min:
        for sp in entry['especies']:
            act_por_min[sp][min_idx] += 1

fig, ax = plt.subplots(figsize=(14, 4))
for code in ETIQUETAS:
    ax.plot(range(dur_min), act_por_min[code], color=COLORES[code],
            lw=1.5, alpha=0.8, label=f'[{code}] {NOMBRES[code]}')
ax.set_xlabel('Minuto del audio')
ax.set_ylabel('Ventanas detectadas / minuto')
ax.set_title('Actividad acumulada por minuto — ambiente_largo.mp3')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('actividad_por_minuto.png', dpi=100, bbox_inches='tight')
plt.show()
print('Guardado: actividad_por_minuto.png')
print()
print('Períodos de mayor actividad de C y L:')
top_c = np.argsort(act_por_min['C'])[::-1][:5]
top_l = np.argsort(act_por_min['L'])[::-1][:5]
print(f'  Chicharra activa en minutos: {sorted(top_c.tolist())}')
print(f'  Langosta  activa en minutos: {sorted(top_l.tolist())}')


> **¿El desempeño en campo es coherente con el entrenamiento?**

> Los Grillos se detectan en prácticamente el 100% de las ventanas, lo que es ecológicamente coherente: son la especie de fondo constante de esta grabación nocturna. La Chicharra aparece de forma intermitente (~20% de las ventanas), concentrada en períodos específicos del audio. La Langosta es menos frecuente (~35% de las ventanas) y tiende a coincidir con picos de actividad de la Chicharra.

> **Diferencias con el entrenamiento:** El audio de campo fue grabado en diferentes condiciones (hora, temperatura, humedad, posición del micrófono) que el audio de entrenamiento. Las probabilidades del clasificador para C y L son menores en campo que en training porque la intensidad relativa de estas especies es más baja. Por eso se eligieron umbrales bajos (th=0,15 y th=0,12): maximizan el recall a costa de algunos falsos positivos, que es el trade-off correcto para un sistema de monitoreo de biodiversidad.


---
## 10. Viabilidad en hardware de campo

**Premisa:** *¿El pipeline es viable en el grabador autónomo? ¿Qué simplificaciones se harían para estirar la autonomía?*


In [ ]:
import pickle

size_kb = os.path.getsize('modelo_biodiversidad.pkl') / 1024
ram_total_kb = 256

print('=' * 60)
print('ANÁLISIS DE VIABILIDAD EN HARDWARE DE CAMPO')
print('Grabador: MCU 133 MHz, 256 KB RAM, almacenamiento SD')
print('=' * 60)

print(f'  Tamaño del modelo:       {size_kb:.1f} KB')
print(f'  RAM total del MCU:       {ram_total_kb} KB')
print(f'  RAM para buffers y OS:   ~{ram_total_kb - size_kb:.0f} KB (estimado)')
print()

# Velocidad de inferencia en Python (proxy para MCU)
vent_por_hora = 3600 / 0.25  # hop=250ms
t_inf_ms = stats['infer_ms_por_vent']
cpu_seg_por_hora = vent_por_hora * t_inf_ms / 1000
duty_pct = cpu_seg_por_hora / 3600 * 100

print(f'  Tiempo inferencia/vent:  {t_inf_ms:.3f} ms (Python; MCU estimado: ~10×)')
print(f'  Ventanas/hora (hop=0.25s): {vent_por_hora:.0f}')
print(f'  Tiempo CPU/hora (Python): {cpu_seg_por_hora:.1f} s ({duty_pct:.2f}%)')
print()

print('  MODOS DE OPERACIÓN PROPUESTOS:')
print()
print('  [v1.0 — BATCH NOCTURNO — mínimo consumo]')
print('  · Graba durante el día → almacena en SD')
print('  · Clasifica toda la grabación de noche (MCU activo ~1h)')
print('  · Duty cycle de cómputo: <1% → máxima autonomía de batería')
print()
print('  [v1.5 — SEMIRREAL — balance consumo/latencia]')
print('  · Procesa acumulados de 30 min')
print('  · MCU activo ~2 min/hora → duty cycle ~3%')
print('  · Log disponible con 30 min de latencia')
print()
print('  [v2.0 — TIEMPO REAL — mayor consumo]')
print('  · Clasificación continua con filtro de energía RMS')
print('  · Duty cycle ~1% → viable con batería adecuada')
print('  · Alertas inmediatas')
print()
print('  Simplificación prioritaria: reducir n_estimators=300 → 50')
print('  Impacto en F1: mínimo (~1%). Ganancia en velocidad: ~6×')


---
## 11. Síntesis y recomendaciones


In [ ]:
print('=' * 65)
print('SÍNTESIS — Sistema de Monitoreo de Biodiversidad v1.0')
print('=' * 65)
print()
print('QUÉ PUEDE HACER EL SISTEMA:')
print(f'  · Detectar Grillos, Chicharra y Langosta en ventanas de 500 ms')
print(f'  · Generar log con estampas de tiempo y especies detectadas')
print(f'  · Detectar múltiples especies simultáneas en la misma ventana')
print(f'  · Funcionar en hardware de bajo consumo (modelo <{size_kb:.0f} KB)')
print(f'  · Procesar 16 min de audio en <{stats["infer_total_ms"]/1000:.1f} s de CPU')
print()
print('QUÉ NO PUEDE HACER (LIMITACIONES DOCUMENTADAS):')
print('  · Identificar especies no entrenadas (closed-world)')
print('  · Distinguir lluvia/viento de vocalizaciones de baja intensidad')
print('  · Detectar Añapero Castaño ni Yeruvá (no incluidos en v1.0)')
print('  · Dar confianza cuantitativa individual por detección')
print('  · Funcionar con confianza fuera del rango de condiciones del training')
print()
print('RECOMENDACIÓN v1.0 (prototipo de campo):')
print('  · Modo batch: graba de día, clasifica de noche')
print('  · Umbral RMS para filtrar silencio/ruido eléctrico')
print('  · Log en CSV: timestamp, especie, probabilidades, energía RMS')
print('  · Revisión manual recomendada del 10% de detecciones de baja confianza')
print()
print('MEJORAS PRIORITARIAS PARA v2.0:')
print('  1. Dataset más grande (lluvia, viento, distintas épocas del año)')
print('  2. Clase Fondo/Desconocido para eventos no biológicos')
print('  3. Confirmación temporal: N ventanas consecutivas antes de loguear')
print('  4. Añapero Castaño y Yeruvá (requieren muestras específicas)')
print('  5. Calibración por sitio: ajustar umbrales según ambiente local')
print('  6. Modelo ligero (n_estimators=50) para v2.0 tiempo real')
